# Build the self-contained DNAmFitAge clock

This notebook embeds the packaged gait, grip, VO2max, and original GrimAge models into one DNAmFitAge artifact. The public inputs are one ordered assay-feature union followed by `female` and `age`; no precomputed `grimage` input is exposed.

## Inputs and provenance

The gait and grip inputs are generated by `dnamfitagegait.ipynb` and `dnamfitagegrip.ipynb`. The retained VO2max and original GrimAge artifacts are the published pyaging components. Semantic digests below pin each component's ordered features, reference values, and learned tensors while deliberately excluding mutable release metadata.

In [1]:
import hashlib
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import pyaging as pya

weights_dir = Path("../weights")
weights_dir.mkdir(parents=True, exist_ok=True)

## Instantiate and describe the clock

In [2]:
# ruff: noqa: E501
model = pya.models.DNAmFitAge()
model.metadata["clock_name"] = "dnamfitage"
model.metadata["data_type"] = "DNA methylation"  # Paper: Blood DNA methylation was used to develop the fitness biomarkers.
model.metadata["species"] = "Homo sapiens"  # Paper: The development cohorts were human adult studies (FHS, BLSA, and Budapest).
model.metadata["year"] = 2023
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "McGreevy, K. M., et al. “DNAmFitAge: biological age indicator incorporating physical fitness.” Aging 15(10): 3904–3938 (2023)."
model.metadata["doi"] = "https://doi.org/10.18632/aging.204538"
model.metadata["notes"] = "Sex-specific Klemera–Doubal biological-age composite that calculates DNAm gait speed, grip strength, VO2max, and original DNAmGrimAge internally from methylation, age, and female inputs."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: The biomarkers were developed from blood DNAm data.
model.metadata["predicts"] = ["physical-fitness biological age"]  # Paper: DNAmFitAge provides an estimate of biological age that incorporates fitness.
model.metadata["training_target"] = ["biological age"]  # Paper: The Klemera–Doubal framework posits an unobserved biological-age trait centered on chronological age.
model.metadata["unit"] = ["years"]  # Paper: FitAgeAcceleration is interpreted as years older or younger than expected chronological age.
model.metadata["model_type"] = "Klemera–Doubal composite"  # Paper: The TrueTrait function carried out the Klemera–Doubal method with sex-specific component weights.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: The reported fitness CpG background and fitted loci were on the 450K array.
model.metadata["population"] = "adults"  # Paper: DNAmFitAge was formed in FHS and BLSA adults and combines sex-appropriate female and male fitness-component estimates.
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 1343
model.metadata["citations"] = 99
model.metadata["citations_date"] = "2026-07-05"

## Load and verify embedded components

In [3]:
def semantic_digest(component):
    digest = hashlib.sha256()
    digest.update(json.dumps(component.features, separators=(",", ":"), allow_nan=True).encode())
    references = (
        component.reference_values.tolist()
        if isinstance(component.reference_values, torch.Tensor)
        else list(component.reference_values)
    )
    digest.update(json.dumps(references, separators=(",", ":"), allow_nan=True).encode())
    for name, tensor in sorted(component.state_dict().items()):
        tensor = tensor.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

In [4]:
model.Gait = torch.load("../weights/dnamfitagegait.pt", weights_only=False, map_location="cpu")
model.Grip = torch.load("../weights/dnamfitagegrip.pt", weights_only=False, map_location="cpu")
model.VO2Max = torch.load("../weights/dnamfitagevo2max.pt", weights_only=False, map_location="cpu")
model.GrimAge = torch.load("../weights/grimage.pt", weights_only=False, map_location="cpu")

component_contracts = {
    "GrimAge": (
        "GrimAge",
        "grimage",
        1032,
        "0e53d4eb90fb41aef954225014554a7ca3484f128be6b4bc85b9180a58186a66",
    ),
    "Gait": (
        "DNAmFitAgeGait",
        "dnamfitagegait",
        111,
        "b95438b4297383243fd919910cf31202e9b65cc419e58415462189aa03dc740f",
    ),
    "Grip": (
        "DNAmFitAgeGrip",
        "dnamfitagegrip",
        183,
        "22f61a810db71482652d79c6551620435f3b1021915453c457cd0e4e157fc7b4",
    ),
    "VO2Max": (
        "DNAmFitAgeVO2Max",
        "dnamfitagevo2max",
        41,
        "75925dbcd35861aee41079c6054652ca05c7854cd13be17f1b0cdd6b5feb7673",
    ),
}
for attribute, (class_name, clock_name, feature_count, expected_digest) in component_contracts.items():
    component = getattr(model, attribute)
    assert type(component).__name__ == class_name
    assert component.metadata["clock_name"] == clock_name
    assert len(component.features) == feature_count
    assert semantic_digest(component) == expected_digest

## Construct the public feature union

First occurrence controls order. Component covariates are removed before unioning and are appended once as the final public fields.

In [5]:
def ordered_union(*groups):
    return list(dict.fromkeys(feature for group in groups for feature in group))

covariates = {"female", "age"}
component_assays = {
    name: [feature for feature in getattr(model, name).features if feature not in covariates]
    for name in component_contracts
}
assay_features = ordered_union(
    component_assays["GrimAge"],
    component_assays["Gait"],
    component_assays["Grip"],
    component_assays["VO2Max"],
)
model.features = assay_features + ["female", "age"]
feature_index = {feature: index for index, feature in enumerate(model.features)}

for name in component_contracts:
    component = getattr(model, name)
    indices = torch.tensor([feature_index[feature] for feature in component.features], dtype=torch.long)
    setattr(model, f"features_{name}", indices)

model.female_index = feature_index["female"]
model.age_index = feature_index["age"]
model.reference_values = [float("nan")] * len(assay_features) + [1.0, 65.0]

In [6]:
assert [len(component_assays[name]) for name in component_contracts] == [1030, 110, 182, 40]
assert len(assay_features) == 1341
assert len(model.features) == 1343
assert len(model.features) == len(set(model.features))
assert model.features[-2:] == ["female", "age"]
assert "grimage" not in model.features
assert model.GrimAge.metadata["clock_name"] == "grimage"
assert model.reference_values[-2:] == [1.0, 65.0]
assert all(math.isnan(value) for value in model.reference_values[:-2])
{name: len(getattr(model, f"features_{name}")) for name in component_contracts}

{'GrimAge': 1032, 'Gait': 111, 'Grip': 183, 'VO2Max': 41}

## Retain the published sex-specific final equations

In [7]:
model.base_model_m = pya.models.LinearModel(input_dim=4)
model.base_model_m.linear.weight.data = torch.tensor(
    [[0.1390346, 0.1787371, 0.1593873, 0.5228411]], dtype=torch.float32
)
model.base_model_m.linear.bias.data = torch.tensor([0.0], dtype=torch.float32)

model.base_model_f = pya.models.LinearModel(input_dim=4)
model.base_model_f.linear.weight.data = torch.tensor(
    [[0.1044232, 0.1742083, 0.2278776, 0.4934908]], dtype=torch.float32
)
model.base_model_f.linear.bias.data = torch.tensor([0.0], dtype=torch.float32)
model.preprocess_name = None
model.preprocess_dependencies = None
model.postprocess_name = None
model.postprocess_dependencies = None

## Validate feature units and deterministic oracles

In [8]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
assert len(model.feature_units) == 1343
pd.DataFrame.from_records(feature_ranges[-2:])

,feature,unit,low,high
0,female,"indicator (1 = female, 0 = male)",0.0,1.0
1,age,years,0.0,122.5


In [9]:
def beta(feature):
    digest = int(hashlib.sha256(feature.encode()).hexdigest()[:8], 16)
    return 0.2 + (digest % 6000) / 10000

frame = pd.DataFrame(
    [{feature: beta(feature) for feature in model.features} for _ in range(2)],
    index=["male", "female"],
)
frame["female"] = [0.0, 1.0]
frame["age"] = 57.0
model.to(torch.float64).eval()
with torch.no_grad():
    predictions = model(torch.as_tensor(frame[model.features].to_numpy(), dtype=torch.float64)).ravel()
np.testing.assert_allclose(
    predictions.numpy(),
    [131.19968704579992, 133.55199017157764],
    rtol=0.0,
    atol=1e-10,
)
predictions

tensor([131.1997, 133.5520], dtype=torch.float64)

In [10]:
missing_covariates = torch.tensor(
    [[beta(feature) for feature in assay_features] + [float("nan"), float("nan")]],
    dtype=torch.float64,
)
filled = model.preprocess(missing_covariates)
assert filled[0, model.female_index].item() == 1.0
assert filled[0, model.age_index].item() == 65.0
with torch.no_grad():
    missing_prediction = model(filled).item()
assert math.isclose(missing_prediction, 137.34972213448359, rel_tol=0.0, abs_tol=1e-10)
missing_prediction

137.34972213448359

## Save the verified artifact

In [11]:
output_path = weights_dir / f"{model.metadata['clock_name']}.pt"
torch.save(model, output_path)
assert output_path.is_file()
{
    "output": str(output_path),
    "features": len(model.features),
    "embedded_components": list(component_contracts),
}

{'output': '../weights/dnamfitage.pt',
 'features': 1343,
 'embedded_components': ['GrimAge', 'Gait', 'Grip', 'VO2Max']}